<a href="https://colab.research.google.com/github/otoperalias/Coyuntura/blob/main/clases/Tema3_V_23-11-2026.ipynb.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

## Tema 3.5. Modelos SARIMA y SARIMAX.

In [ ]:
# Importamos funciones y establecemos configuración general
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import statsmodels.api as sm
from statsmodels.graphics.tsaplots import plot_acf, plot_pacf
from statsmodels.tsa.stattools import adfuller
plt.style.use('seaborn-v0_8-whitegrid')
plt.rcParams["figure.figsize"] = [10,4]  # Default figure size

In [ ]:
# Generamos funciones que vamos a usar en el notebook
def DFtest(datos):
    print('Results of Dickey-Fuller Test (H0= NO estac.):')
    dftest = adfuller(datos, autolag='AIC')
    print('Test Statistic', dftest[0])
    print('p-value', dftest[1])
    print('#Lags Used', dftest[2])
    print('# obs', dftest[3])
    for k, v in dftest[4].items():
        print(k,v)

def forecast_accuracy(forecast, actual):
    rmse = np.sqrt(np.mean((forecast-actual)**2))
    mape = np.mean(np.abs(forecast - actual)/np.abs(actual))  # MAPE
    corr = np.corrcoef(forecast, actual)[0,1]   # corr
    print({'rmse':rmse,'mape':mape,'corr':corr})

### Modelo SARIMA

In [ ]:
# Importamos base de datos (sobre ventas de medicamentos)
# https://github.com/otoperalias/Coyuntura/blob/main/clases/datos/medicamentos_ventas.csv
# clicar en 'raw' y luego clicar en botón derecho y 'guardar como' .csv
#medic = pd.read_csv("medicamentos_ventas.csv",index_col=0,parse_dates=True)

# O directamente:
medic = pd.read_csv("https://raw.githubusercontent.com/otoperalias/Coyuntura/main/clases/datos/medicamentos_ventas.csv",index_col=0,parse_dates=True)

#Creamos test and training dataset
train=medic[:-24]
test=medic[-24:]

In [ ]:
# GRÁFICO
fig, axes = plt.subplots(2, 1, figsize=(8,5), dpi=100, sharex=True)
# Diferenciación normal
axes[0].plot(train, label='Serie original')
axes[0].plot(train.diff(1), label='Diferenciación normal')
axes[0].set_title('Diferenciación normal')
axes[0].legend(loc='upper left', fontsize=10)
# Diferenciación estacional
axes[1].plot(train, label='Serie original')
axes[1].plot(train.diff(12), label='Diferenciación estacional', color='green')
axes[1].set_title('Diferenciación estacional')
plt.legend(loc='upper left', fontsize=10)
plt.suptitle('MEDICAMENTOS - VENTAS', fontsize=13)
plt.show()

Se puede observar que los picos estacionales no resultan afectados cuando se realiza la diferenciación normal, mientras que al diferenciar de manera estacional, desaparecen.

In [ ]:
# Dibujamos correlogramas y usamos el test DF
fig, ax = plt.subplots(2, figsize=(12,6))
ax[0] = plot_acf(train.diff(12).dropna(), ax=ax[0], lags=28)
ax[1] = plot_pacf(train.diff(12).dropna(), ax=ax[1], lags=28)
plt.show()
DFtest(train.diff(12).dropna())

Se rechaza la hipótesis nula y, por tanto, la serie con diferencias estacionales es estacionario.  
En caso de que la serie con diferencias estacionales fuera NO estacionaria, habría que tomar, a su vez, primeras diferencias: ```train.diff(12).diff()```.


Según el los gráficos de autocorrelación y correlación parcial, vemos que el modelo podría ser: SARIMA(2,0,2)(0,1,0,12)

#### Procedemos a construir el modelo:
Para más información sobre modelos SARIMAX:
*https://www.statsmodels.org/stable/generated/statsmodels.tsa.statespace.sarimax.SARIMAX.html
*https://www.statsmodels.org/stable/examples/notebooks/generated/statespace_sarimax_stata.html


In [ ]:
# Especificación del modelo
mod = sm.tsa.statespace.SARIMAX(train,order=(2,0,1),seasonal_order=(0,1,0,12), trend="c",
                                enforce_stationarity=False, enforce_invertibility=False)

In [ ]:
# Estimación del modelo y output de los resultados:v
results= mod.fit()
print(results.summary())   # vemos que la constante no es significativa y se podría quitar

In [ ]:
# Podemos dibujar los errores
results.plot_diagnostics()
plt.show()

#### ¿Cómo de preciso es nuestro modelo para predecir?

In [ ]:
# Realizo la predicción para el periodo de la "test dataset" y calculo su exactitud (acccuracy)
pred = results.get_prediction('2006-07-01','2008-06-01', dynamic=False)
forecast_accuracy(pred.predicted_mean, test.value)

In [ ]:
# Dibujamos la predicción:
pred_ci = pred.conf_int()
# Creando las series para su representación gráfica
fitted_series = pred.predicted_mean
lower_series = pred_ci["lower value"]
upper_series = pred_ci["upper value"]
# Gráfico -
fig,ax=plt.subplots(figsize=(12,8))
ax.plot(train[-60:], label="Training period", color="b") #con [-60:] evito representar la serie completa
ax.plot(test, label="Actual values", color="k")
ax.plot(fitted_series, label="Predicted", color='r')
ax.fill_between(lower_series.index,
                 lower_series,
                 upper_series,
                 color='k', alpha=.15)
ax.legend()
ax.set_title("SARIMA - Predicción de ventas de medicamentos")
plt.show()

In [ ]:
# Y también predecimos hacia el futuro
results = sm.tsa.statespace.SARIMAX(medic,order=(2,0,1),seasonal_order=(0,1,0,12), trend="c",
                                enforce_stationarity=False, enforce_invertibility=False).fit()
pred = results.get_prediction(start="2008-07-01",end="2010-06-01")
pred_ci = pred.conf_int()
# Creando las series para su representación gráfica
fitted_series = pred.predicted_mean
lower_series = pred_ci["lower value"]
upper_series = pred_ci["upper value"]
# Gráfico
fig,ax=plt.subplots(figsize=(12,8))
ax.plot(medic)
ax.plot(fitted_series, color='darkgreen')
ax.fill_between(lower_series.index,
                 lower_series,
                 upper_series,
                 color='k', alpha=.15)
ax.set_title("SARIMA - Predicción de ventas de medicamentos")
plt.show()

### Modelo SARIMAX

La predicción del modelo SARIMA es mejorable (*rmse*=3.31, *mape*= 0.13, *corr*= 0.81) y nos gustaría que fuera más buena.  
Si tuviéramos un indicador correlacionado con las ventas de medicamentos y que estuviera disponible antes, sería muy útil para ayudarnos a predecir. Ese indicador sería la variable e**X**ógena del modelo *SARIMAX*.

In [ ]:
# Datos de la variable exógena: número de visitas a atención primaria
# https://github.com/otoperalias/Coyuntura/blob/main/clases/datos/medic_visitas_doctor.xlsx

# visit = pd.read_excel("medic_visitas_doctor.xlsx", index_col=0, parse_dates=True)

# O directamente leemos del url:
visit = pd.read_excel("https://github.com/otoperalias/Coyuntura/raw/refs/heads/main/clases/datos/medic_visitas_doctor.xlsx", index_col=0, parse_dates=True)

visit

In [ ]:
# Dividir la serie en c subseries

x_train=visit.loc["1991-07-01":"2006-06-01"]
x_test=visit.loc["2006-07-01":"2008-06-01"]
x_out=visit.loc["2008-07-01":"2008-12-01"]
print(len(x_train))
print(len(x_test))
print(len(x_out))

Esta serie tienes 6 observaciones más, las cuales vamos utilizar para mejorar la predicción.

In [ ]:
# Visualizamos
fig,ax=plt.subplots(1,2,figsize=(10,4))
ax[0].plot(medic)
ax01 = ax[0].twinx()  # aquí se establece un segundo eje Y que comparte el mismo eje X
ax01.plot(visit,color='tab:red')
ax01.grid(False)
ax[1].scatter(medic.value,visit.visitas[:204]) # vemos que hay una correlación alta y por tanto puede usarse como variable exógena
plt.subplots_adjust(wspace=0.5)
plt.show()

#Calculamos correlación para la train
print(np.corrcoef(train.value,x_train.visitas)[0,1])

Vemos que las dos series están muy correlacionadas y por tanto la variable visitas puede usarse como eXógena.

#### Construimos el modelo:

In [ ]:
# Especificamos y estimamos el modelo
results = sm.tsa.statespace.SARIMAX(train,exog=x_train, order=(2,0,1),seasonal_order=(0,1,0,12), trend="c",
                                    enforce_stationarity=False, enforce_invertibility=False).fit()
print(results.summary())

#### Capacidad para predecir:

In [ ]:
# Capacidad para predecir
pred = results.get_prediction('2006-07-01','2008-06-01', exog=x_test, dynamic=False)
forecast_accuracy(pred.predicted_mean, test.value)

In [ ]:
# Dibujamos la predicción:
pred_ci = pred.conf_int()
# Creando las series para su representación gráfica
fitted_series = pred.predicted_mean
lower_series = pred_ci["lower value"]
upper_series = pred_ci["upper value"]
# Gráfico -
fig,ax=plt.subplots(figsize=(12,8))
ax.plot(train[-60:], label="Training period", color="b") #con [-60:] evito representar la serie completa
ax.plot(test, label="Actual values", color="k")
ax.plot(fitted_series, label="Predicted", color='r')
ax.fill_between(lower_series.index,
                 lower_series,
                 upper_series,
                 color='k', alpha=.15)
ax.legend()
ax.set_title("SARIMAX - Predicción de ventas de medicamentos \n usando visitas al doctor como variable exógena")
plt.show()

Y también predecimos hacia el futuro. Dado que visit llega a 2008-12-01 (mientras medic hasta 2008-06-01), predecimos seis meses.

In [ ]:
# Predecimos hacia el futuro:
exog_in=visit[:-6]
results = sm.tsa.statespace.SARIMAX(medic,exog=exog_in, order=(2,0,1),seasonal_order=(0,1,0,12), trend="c",
                                    enforce_stationarity=False, enforce_invertibility=False).fit()
pred = results.get_prediction(start="2008-07-01",end="2008-12-01", exog=x_out)
pred_ci = pred.conf_int()
# Creando las series para su representación gráfica
fitted_series = pred.predicted_mean
lower_series = pred_ci["lower value"]
upper_series = pred_ci["upper value"]
# Gráfico
fig,ax=plt.subplots(figsize=(12,8))
ax.plot(medic[-60:])
ax.plot(fitted_series, color='darkgreen')
ax.fill_between(lower_series.index,
                 lower_series,
                 upper_series,
                 color='k', alpha=.15)
ax.set_title("SARIMAX - Predicción de ventas de medicamentos \n usando visitas al doctor como variable exógena")
plt.show()

## Ejercicio: modelización del crecimiento del PIB interanual de España con el modelo SARIMA  y SARIMAX.  

Datos a utilizar:
https://github.com/otoperalias/Coyuntura/blob/main/clases/datos/Clase_SarimaxDatos.xlsx

In [ ]:
#Importamos y procesamos
pib=pd.read_excel("/content/Clase_SarimaxDatos.xlsx",sheet_name="PIB INE-CNTR")
pib=pib.iloc[3:129]
pib.columns=["periodo","ive"]
pib['periodo']=pib.periodo.str[:4]+"Q"+pib.periodo.str[-1:]
pib['periodo']=pd.to_datetime(pib['periodo'])
pib.set_index("periodo",inplace=True)
pib=pib.astype(float)

# Train/test split
train=pib.loc["1995-01-01":"2024-04-01"]
test=pib.loc["2024-07-01":"2026-07-01"]
print(len(train),len(test))


In [ ]:
# GRÁFICO
fig, axes = plt.subplots(2, 1, figsize=(8,5), dpi=100, sharex=True)
# Diferenciación normal
axes[0].plot(train, label='Serie original')
axes[0].plot(train.diff(1), label='Diferenciación normal')
axes[0].set_title('Diferenciación normal')
axes[0].legend(loc='upper left', fontsize=10)
# Diferenciación estacional
axes[1].plot(train, label='Serie original')
axes[1].plot(train.diff(4), label='Diferenciación estacional', color='green')
axes[1].set_title('Diferenciación estacional')
plt.legend(loc='upper left', fontsize=10)
plt.suptitle('PIB', fontsize=13)
plt.show()

In [ ]:
# Dibujamos correlogramas y usamos el test DF
fig, ax = plt.subplots(2, figsize=(12,6))
ax[0] = plot_acf(train.diff(4).dropna(), ax=ax[0], lags=28)
ax[1] = plot_pacf(train.diff(4).dropna(), ax=ax[1], lags=28)
plt.show()
DFtest(train.diff(4).dropna())

In [ ]:
mod = sm.tsa.statespace.SARIMAX(train,order=(1,0,0),seasonal_order=(1,1,0,4), trend="c",
                                enforce_stationarity=False, enforce_invertibility=False)
# Estimación del modelo y output de los resultados:v
results= mod.fit()
#print(results.summary())   # vemos que la constante no es significativa y se podría quitar

In [ ]:
# Realizo la predicción para el periodo de la "test dataset" y calculo su exactitud (acccuracy)
pred = results.get_prediction('2024-07-01','2026-04-01', dynamic=False)
forecast_accuracy(pred.predicted_mean, test.ive)

In [ ]:
# Dibujamos la predicción:
pred_ci = pred.conf_int()
# Creando las series para su representación gráfica
fitted_series = pred.predicted_mean
lower_series = pred_ci["lower ive"]
upper_series = pred_ci["upper ive"]
# Gráfico -
fig,ax=plt.subplots(figsize=(12,8))
ax.plot(train, label="Training period", color="b") #con [-60:] evito representar la serie completa
ax.plot(test, label="Actual values", color="k")
ax.plot(fitted_series, label="Predicted", color='r')
ax.fill_between(lower_series.index,
                 lower_series,
                 upper_series,
                 color='k', alpha=.15)
ax.legend()
ax.set_title("SARIMA - Predicción del PIB")
plt.show()

In [ ]:
# Y también predecimos hacia el futuro
mod = sm.tsa.statespace.SARIMAX(pib,order=(1,0,0),seasonal_order=(1,1,0,4), trend="c",
                                enforce_stationarity=False, enforce_invertibility=False)
pred = results.get_prediction(start="2026-07-01",end="2028-04-01")
pred_ci = pred.conf_int()
# Creando las series para su representación gráfica
fitted_series = pred.predicted_mean
lower_series = pred_ci["lower ive"]
upper_series = pred_ci["upper ive"]
# Gráfico
fig,ax=plt.subplots(figsize=(12,8))
ax.plot(pib)
ax.plot(fitted_series, color='darkgreen')
ax.fill_between(lower_series.index,
                 lower_series,
                 upper_series,
                 color='k', alpha=.15)
ax.set_title("SARIMA - Predicción del PIB")
plt.show()

In [ ]:
# Ahora, en tasas de crecimiento

fitted_series.rename("ive", inplace=True)
#Creación serie enlazada
serie_enlazada=pd.concat([pib,fitted_series])

fig,ax=plt.subplots(figsize=(12,8))
ax.plot(serie_enlazada["2022":].pct_change(4))
ax.plot(serie_enlazada["2025-04-01":].pct_change(4), color="tab:red")

ax.set_title("SARIMA - Predicción de la variación interanual del PIB")
plt.show()

In [ ]:
# ESTIMACIÓN SARIMA CON TASA DE VARIACIÓN INTERANUAL

#Importamos y procesamos
tvpib=pib.pct_change(4)

# Train/test split
train=tvpib.loc["1996-01-01":"2024-04-01"]  # Ahora empezamos en el 96, pues para el 95 no se puede calcular
test=tvpib.loc["2024-07-01":"2026-07-01"]
print(len(train),len(test))

In [ ]:
train.plot()

In [ ]:
# Dibujamos correlogramas y usamos el test DF
fig, ax = plt.subplots(2, figsize=(12,6))
ax[0] = plot_acf(train.dropna(), ax=ax[0], lags=28)
ax[1] = plot_pacf(train.dropna(), ax=ax[1], lags=28)
plt.show()
DFtest(train.dropna())

In [ ]:
mod = sm.tsa.statespace.SARIMAX(train,order=(1,0,2),seasonal_order=(1,0,0,4), trend="c",
                                enforce_stationarity=False, enforce_invertibility=False)
# Estimación del modelo y output de los resultados:v
results= mod.fit()
print(results.summary())   # vemos que la constante no es significativa y se podría quitar

In [ ]:
pred = results.get_prediction('2024-07-01','2026-04-01', dynamic=False)
forecast_accuracy(pred.predicted_mean, test.ive)

In [ ]:
# Dibujamos la predicción:
pred_ci = pred.conf_int()
# Creando las series para su representación gráfica
fitted_series = pred.predicted_mean
lower_series = pred_ci["lower ive"]
upper_series = pred_ci["upper ive"]
# Gráfico -
fig,ax=plt.subplots(figsize=(12,8))
ax.plot(train, label="Training period", color="b") #con [-60:] evito representar la serie completa
ax.plot(test, label="Actual values", color="k")
ax.plot(fitted_series, label="Predicted", color='r')
ax.fill_between(lower_series.index,
                 lower_series,
                 upper_series,
                 color='k', alpha=.15)
ax.legend()
ax.set_title("SARIMA - Predicción de la tasas de variación del PIB")
plt.show()

In [ ]:
# Y también predecimos hacia el futuro
mod = sm.tsa.statespace.SARIMAX(tvpib,order=(1,0,2),seasonal_order=(1,0,0,4), trend="c",
                                enforce_stationarity=False, enforce_invertibility=False)
pred = results.get_prediction(start="2026-07-01",end="2028-04-01")
pred_ci = pred.conf_int()
# Creando las series para su representación gráfica
fitted_series = pred.predicted_mean
lower_series = pred_ci["lower ive"]
upper_series = pred_ci["upper ive"]
# Gráfico
fig,ax=plt.subplots(figsize=(12,8))
ax.plot(tvpib["2023":])
ax.plot(fitted_series["2023":], color='darkgreen')
ax.fill_between(lower_series["2023":].index,
                 lower_series["2023":],
                 upper_series["2023":],
                 color='k', alpha=.15)
ax.set_title("SARIMA - Predicción del PIB")
plt.show()

In [ ]:
# Importamos los datos de afiliaciones y consumo eléctrico
inds=pd.read_excel("/content/Clase_SarimaxDatos.xlsx", sheet_name="BDSICE SS Y GWH")
inds.columns=["periodo","ss","gwh"] # Renombro columnas
inds["periodo"]=inds["periodo"].astype(str)  # Convierto columna de fecha en string
inds["periodo"]=inds["periodo"].str[:4]+"-"+inds["periodo"].str[-2:]  # Transformo columnas de fecha en formato reconocible por pandas
inds["periodo"]=pd.to_datetime(inds["periodo"]) # Convierto en formato datetime
inds.set_index("periodo", inplace=True) # Establezco columna de fecha como index

In [ ]:
# Trimestralizar > Tasas de variación > train/test split > comparación

# Trimestralizar
inds = inds.resample("QS").mean()

# Tasas de variación
tvinds = inds.pct_change(4)

# Train/test split
tvinds_train=tvinds.loc["1996-01-01":"2024-04-01"]  # Ahora empezamos en el 96, pues para el 95 no se puede calcular
tvinds_test=tvinds.loc["2024-07-01":"2026-04-01"]
tvinds_pib=tvinds.loc["1996-01-01":"2026-04-01"] # Periodo completo para el que tenemos datos de PIB
tvinds_out=tvinds.loc["2026-07-01":"2026-07-01"] # Timestre extra para el que tenemos las variables exógenas
print(len(train),len(test),len(tvinds_pib), len(tvinds_out))
# Correlaciones
print("Correlación crec PIB - Crec Afiliaciones: ", np.corrcoef(train.ive, tvinds_train['ss'])[0,1])
print("Correlación crec PIB - Crec Consumo eléct: ",np.corrcoef(train.ive, tvinds_train['gwh'])[0,1])


In [ ]:
fig,ax=plt.subplots(1,2,figsize=(10,4))
ax[0].plot(train, label="PIB")
ax01 = ax[0].twinx()  # aquí se establece un segundo eje Y que comparte el mismo eje X
ax01.plot(tvinds_train['ss'],color='tab:red', label="Afiliaciones")
ax01.grid(False)
ax[0].legend()
ax01.legend()
ax[1].scatter(train.ive, tvinds_train['ss']) # vemos que hay una correlación alta y por tanto puede usarse como variable exógena
plt.subplots_adjust(wspace=0.5)
plt.show()

In [ ]:
# Especificamos y estimamos el modelo
results = sm.tsa.statespace.SARIMAX(train, exog=tvinds_train['ss'],order=(1,0,2),seasonal_order=(1,0,0,4), trend="c",
                                    enforce_stationarity=False, enforce_invertibility=False).fit()
print(results.summary())

In [ ]:
# Capacidad para predecir
pred = results.get_prediction('2024-07-01','2026-04-01', exog=tvinds_test['ss'], dynamic=False)
forecast_accuracy(pred.predicted_mean, test.ive)

In [ ]:
# Dibujamos la predicción:
pred_ci = pred.conf_int()
# Creando las series para su representación gráfica
fitted_series = pred.predicted_mean
lower_series = pred_ci["lower ive"]
upper_series = pred_ci["upper ive"]
# Gráfico -
fig,ax=plt.subplots(figsize=(12,8))
ax.plot(train, label="Training period", color="b") #con [-60:] evito representar la serie completa
ax.plot(test, label="Actual values", color="k")
ax.plot(fitted_series, label="Predicted", color='r')
ax.fill_between(lower_series.index,
                 lower_series,
                 upper_series,
                 color='k', alpha=.15)
ax.legend()
ax.set_title("SARIMA - Predicción de la tasas de variación del PIB")
plt.show()

In [ ]:
# Y también predecimos hacia el futuro
results = sm.tsa.statespace.SARIMAX(tvpib.loc["1996-01-01":"2026-04-01"], exog=tvinds_pib['ss'], order=(1,0,2),seasonal_order=(1,0,0,4), trend="c",
                                enforce_stationarity=False, enforce_invertibility=False).fit()
pred = results.get_prediction(start="2026-07-01",end="2026-07-01", exog=tvinds_out['ss'], dynamic=False)
pred_ci = pred.conf_int()
# Creando las series para su representación gráfica
fitted_series = pred.predicted_mean
lower_series = pred_ci["lower ive"]
upper_series = pred_ci["upper ive"]
# Gráfico
fig,ax=plt.subplots(figsize=(12,8))
ax.plot(tvpib["2023":])
ax.plot(fitted_series[-1:], marker="o", color='darkgreen')
ax.plot(lower_series.index[-1],lower_series[-1], marker="_", color='darkgreen')
ax.plot(lower_series.index[-1], upper_series[-1], marker="_", color='darkgreen')
ax.set_title("SARIMAX - Predicción del PIB")
plt.show()

## Modelización y predicción de la tasa de paro

El ejercicio consiste en modelizar con el SARIMA y SARIMAX la tasa de paro hasta el segundo trimestre del 2026 y predecir el tercer trimestre usando las afiliaciones como variable exógena.

In [ ]:
# Importar datos y procesarlos
tp=pd.read_excel("/content/65219.xlsx")
tp=tp.iloc[7:106]
tp.columns=["periodo","tasa"]
tp['periodo']=tp.periodo.str[:4]+"Q"+tp.periodo.str[-1:]
tp['periodo']=pd.to_datetime(tp['periodo'])
tp.set_index("periodo",inplace=True)
tp=tp.astype(float)

# Train/test split
train=tp.loc["2002-01-01":"2024-04-01"]
test=tp.loc["2024-07-01":"2026-07-01"] # (el dato del tercer trimestre de 2026 lo dejamos fuera)
print(len(train),len(test))

**HOJA DE RUTA**

-- Con la train dataset --

* Analizar serie y su correlograma
* Test de raíz unitaria
* Decidir si tomar diferencias estacionales y/o primeras diferencias
* Mostrar otra vez los gráficos de correlogramas, con la serie transformada.
* Decidir los parámetros del modelo
* Estimar el modelo y ver el resumen del mismo

-- Out of sample validation --

* Predecir con el modelo para el periodo de la test dataset
* Calcular métricas de bondad de la predicción comparando la predicción con la test dataset
* Visualizar la predicción, incluyendo los intervalos de confianza.

-- Predecir hacia el futuro ---

* Reestimar el modelo con la serie completa (train + test), manteniendo la misma especificación
* Predecir el tercer trimestre de 2026.
* Visualizar la predicción, incluyendo los intervalos de confianza.

-- Tareas adicionales con SARIMAX --

* Comprobar la correlación entre la variable a predecir y la exógena (con la train dataset)
* Repetir los pasos anteriores incluyendo la variable exógena (con igual periodo que la serie)
* Comparar las métricas de bondad de la predicción del SARIMA y del SARIMAX

-- Comparar predicción del tercer trimestre con los datos reales --
* Comparar las predicciones del tercer trimestre de los modelos SARIMA y SARIMAX con los datos reales